# Lesson 28 · Mean, variance, standardization — making apples and oranges comparable with z-scores

**Learning goals**
1. Compute the mean / variance / standard deviation, and use z-scores to bring data in different units onto the same scale
2. Know why you can't compare "apples and oranges" directly: standardize first, then compare shape and location
3. Carry the same intuition over to MFCC frame features: before they go into a model, they usually get a per-dimension z-score

> **One engineering note, not a statistics course**: population variance divides by $n$, sample variance by $n-1$ — a conventional difference that matters when "estimating a population". In this lesson, just follow the course's convention in the implementation and the assertions.

**Why it matters for Aurora**: MFCC features usually get a `zscore` before they're fed to a model; you can think of BatchNorm / LayerNorm as "a z-score with learnable scaling". Skip putting things on a common scale, and gradients are much more likely to explode or vanish.

← **Previous lesson** · [L27 · Probability basics](L27_probability_basics.ipynb)

> Last lesson covered **probability basics**: events, conditional probability, independence and the law of large numbers.
> This lesson explores **mean, variance, standardization**.

## This lesson's story: how do audio features compare "apples and oranges"?

Record audio on different microphones and the MFCC features land in completely different numerical ranges — one might be [-50, 50], another [-5, 5]. Feed them straight into a similarity calculation and the one with bigger numbers "dominates" the result, which isn't fair.

The fix: **z-score standardization**

```
z = (x - μ) / σ
```

It brings every feature into the same "language": mean = 0, standard deviation = 1. Whatever the original units, the standardized values can be compared directly.

Three core statistics:
- **mean μ**: the data's center of mass, `x.mean()`
- **standard deviation σ**: the typical distance of the data from that center, `x.std()`
- **z-score z**: how many standard deviations the current value is from the mean, `(x - μ) / σ`

**A concrete example**: x = [2, 4, 4, 4, 5, 5, 7, 9]
- μ = 40/8 = 5
- σ² = [(2-5)²+...+(9-5)²]/8 = 32/8 = 4  →  σ = 2
- the z-score of x[0]=2 = (2-5)/2 = -1.5 (1.5 standard deviations below the mean)

In this lesson you implement `zscore(x)` and verify that after standardizing, mean ≈ 0 and standard deviation ≈ 1.

## 1. The geometric intuition behind the three statistics

**Mean μ**: imagine putting the "weight" of every data point on a number line; the mean is the balance point.

$$\mu = \frac{1}{N} \sum_{i=1}^N x_i$$

**Variance (σ²)**: the average of the squared distances from each point to the mean — the bigger the distances, the more "spread out" the distribution.

$$\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2$$

**Standard deviation σ = √σ²**: in the same units as the original data, so easier to read.

**Worked by hand**: x = [2, 4, 4, 4, 5, 5, 7, 9]

```
μ = (2+4+4+4+5+5+7+9)/8 = 40/8 = 5
σ² = ((2-5)²+(4-5)²×3+(5-5)²×2+(7-5)²+(9-5)²)/8
   = (9+1+1+1+0+0+4+16)/8 = 32/8 = 4
σ = √4 = 2
```

**Check after the z-score**: the standardized array should have mean 0 and standard deviation 1 (to machine precision).

> **In NumPy**: `x.mean()` = μ, `x.std()` = σ (population standard deviation by default, ddof=0).
> For the sample standard deviation use `x.std(ddof=1)` — the n-1 correction (Bessel's correction).

### 🔍 Sample vs population standard deviation (the ddof parameter)

Above we said "for the sample standard deviation use `x.std(ddof=1)`"; here's why in detail.

**Background**:
- **population standard deviation** $\sigma = \sqrt{\frac{1}{N}\sum(x_i-\mu)^2}$ — computed over the whole population
- **sample standard deviation** $S = \sqrt{\frac{1}{N-1}\sum(x_i-\bar{x})^2}$ — estimates the population parameter from a sample

Note the different denominators: N vs N-1.

**Why does a sample use N-1? — Bessel's correction**

The correction is called "Bessel's correction" after the mathematician Friedrich Wilhelm Bessel, who introduced it in the 19th century. The reason:

When we compute a sample's variance, the sample mean $\bar{x}$ has already been computed from the data itself as its "best fit", which makes the sample variance **underestimate** the true population variance. Using N-1 instead of N as the denominator compensates for that underestimate.

An intuitive example:

```python
x = np.array([1, 2, 3, 4, 5])   # the whole population, N=5

# population standard deviation (assume these 5 numbers are all the data)
sigma = x.std()           # ddof=0, i.e. the 1/5 formula
# → √[(1-3)² + (2-3)² + (3-3)² + (4-3)² + (5-3)²] / 5 = √(10/5) = √2 ≈ 1.41

# sample standard deviation (assume these 5 numbers are just a sample, used to estimate the population)
s = x.std(ddof=1)         # the 1/4 formula
# → √[...] / 4 = √(10/4) = √2.5 ≈ 1.58  (larger, compensating for the underestimate)
```

**When to use ddof=0 and when ddof=1?**

A simple rule:
- **the data is the whole population** → use `ddof=0` (e.g. the heights of every student in one class)
- **the data is a sample drawn from a population** → use `ddof=1` (e.g. pick 30 students at random from across the country and use their heights to estimate the national average)

**How Aurora uses it**:
z-score standardization usually uses `ddof=0` (i.e. `.std()`), because we're working with the current batch of features (treated as the complete "population"), not estimating the parameters of some larger population.

### 🤔 Why does variance use the "square" and not the "absolute value"?

A really sharp question. Intuitively, we could just as well measure "spread" with `mean absolute distance = (1/N)∑|x_i - μ|`. So why did statisticians choose the square?

**Three reasons**:

1. **It's easy to work with mathematically**
   The square function $f(x) = x^2$ is differentiable everywhere, and its extrema are easy to find. Differentiating:
   - the derivative of $x^2$ → $2x$, clean
   - the derivative of $|x|$ → not differentiable at $x=0$, defined piecewise, messy to analyze

   That makes variance easier to handle in optimization algorithms and statistical inference.

2. **"Big deviations" are penalized more**
   Suppose `x = [1, 5]`, μ = 3:
   - with absolute values: `|1-3| + |5-3| = 4`
   - with squares: `(1-3)² + (5-3)² = 8`

   If a value is especially far from the mean (an outlier, say), squaring "amplifies" its influence. That's handy for anomaly detection.

3. **A deep link with the normal distribution** (advanced; just be aware of it for now)
   If the data come from a normal distribution, the square-based variance is the optimal unbiased estimator. Statistics has a rigorous proof of this.

**The trade-off**: the absolute-value version, the "mean absolute deviation (MAD)", is more robust in some settings (less sensitive to outliers), but variance is used far more widely because of its nice mathematical properties.

### 📐 An algebraic proof of z-score standardization: why mean = 0 and standard deviation = 1?

This one deserves a careful answer. Let's derive it rigorously.

**Step 1: show that after subtracting the mean, the new mean is 0**

Let the original data be $x_1, x_2, ..., x_N$, with mean $\mu$.

The first step of standardizing is $x'_i = x_i - \mu$. The mean of the new data:

$$E[x'] = \frac{1}{N}\sum_{i=1}^{N}(x_i - \mu) = \frac{1}{N}\sum_{i=1}^{N}x_i - \frac{1}{N}\sum_{i=1}^{N}\mu$$

$$= \frac{1}{N}\sum_{i=1}^{N}x_i - \frac{N \cdot \mu}{N} = \mu - \mu = 0$$

✅ So after subtracting the mean, the new data's mean is automatically 0.

**Step 2: show that after dividing by the standard deviation, the new standard deviation is 1**

After subtracting the mean, the data is $x'_i = x_i - \mu$, and its standard deviation is still the original $\sigma$.

Then standardize, $z_i = \frac{x'_i}{\sigma} = \frac{x_i - \mu}{\sigma}$. The variance of the new data:

$$\sigma_z^2 = \frac{1}{N}\sum_{i=1}^{N}z_i^2 = \frac{1}{N}\sum_{i=1}^{N}\left(\frac{x_i - \mu}{\sigma}\right)^2$$

$$= \frac{1}{N\sigma^2}\sum_{i=1}^{N}(x_i - \mu)^2 = \frac{1}{\sigma^2} \cdot \frac{1}{N}\sum_{i=1}^{N}(x_i - \mu)^2$$

$$= \frac{1}{\sigma^2} \cdot \sigma^2 = 1$$

The new data's standard deviation is $\sigma_z = \sqrt{\sigma_z^2} = \sqrt{1} = 1$.

✅ So after dividing by the original standard deviation, the new data's standard deviation is automatically 1.

**Checking with concrete numbers**: `x = [2, 4, 4, 4, 5, 5, 7, 9]`, μ=5, σ=2

```
z = (x - 5) / 2 = [-1.5, -0.5, -0.5, -0.5, 0, 0, 1, 2]
mean of z = (-1.5 - 0.5 - 0.5 - 0.5 + 0 + 0 + 1 + 2) / 8 = 0 / 8 = 0 ✓
variance of z = (2.25 + 0.25 + 0.25 + 0.25 + 0 + 0 + 1 + 4) / 8 = 8 / 8 = 1 ✓
standard deviation of z = √1 = 1 ✓
```

### 💡 A hand-calculation trick: grouping repeated values

When computing a variance by hand, if the data has repeated values you can simplify with "multiplication".

Take `x = [2, 4, 4, 4, 5, 5, 7, 9]` with μ=5, and compute $\sigma^2 = \frac{1}{8}\sum(x_i - 5)^2$

**The long way** (term by term):
$$\sigma^2 = \frac{1}{8}[(2-5)^2 + (4-5)^2 + (4-5)^2 + (4-5)^2 + (5-5)^2 + (5-5)^2 + (7-5)^2 + (9-5)^2]$$

**The short way** (grouped by value):
since 4 appears 3 times and 5 appears twice, you can write directly:
$$\sigma^2 = \frac{1}{8}[(2-5)^2 \times 1 + (4-5)^2 \times 3 + (5-5)^2 \times 2 + (7-5)^2 \times 1 + (9-5)^2 \times 1]$$

$$= \frac{1}{8}[9 \times 1 + 1 \times 3 + 0 \times 2 + 4 \times 1 + 16 \times 1] = \frac{1}{8}[9 + 3 + 0 + 4 + 16] = \frac{32}{8} = 4$$

This is **weighting by frequency** — if a value repeats k times, its contribution to the variance gets multiplied by k. It's perfectly correct mathematically, because the associative and commutative laws of addition let us group terms this way.

## Into the lab: computing mean, variance and standard deviation from an array

Call `x.mean()`, `x.var()` and `x.std()` directly on the 8 numbers `[2, 4, 4, 4, 5, 5, 7, 9]`; notice the numerical relationship `var == std²`, and how the mean equals the arithmetic average of all the values.

### 📚 Good to know: NumPy random numbers with `rng.normal()`

The code ahead uses `rng.normal(μ, σ, size=n)` to sample from the **normal distribution** (the Gaussian). The normal distribution is covered properly in L29, but here's what the parameters mean, so the code doesn't look like magic.

```python
rng.normal(0, 1, size=100)    # sample 100 numbers from a normal distribution with mean=0, std=1
rng.normal(5, 2, size=100)    # sample 100 numbers from a normal distribution with mean=5, std=2
```

Parameter order: `normal(loc=mean, scale=standard deviation, size=number of samples)`.

**Intuition**: the normal distribution is "nature's most common bell-shaped distribution" (people's heights and measurement errors, for example, are close to normal). The parameter loc sets where the center of the bell is, and scale sets how wide the bell is.

In the code that follows we'll use it again and again to simulate real data, and watch how the statistics (mean, standard deviation) settle down as the sample size grows.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print('mean           :', x.mean())
print('variance var   :', x.var())
print('std. dev. std  :', x.std())

## Hands-on: sample size and the stability of the sample standard deviation

At n=10 the estimated sample mean and std swing a lot; at n=10000 they approach the population parameters (μ=0, σ=1). Compare the three sample sizes and watch the statistics converge.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    sample = rng.normal(0, 1, size=n)   # sample from the standard normal
    print(f'n={n:6d}  sample mean={np.mean(sample):+.3f}  sample std={np.std(sample):.3f}')


## Code experiment: repeat 5 times and watch the jitter of the sample std shrink with n

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        sample = rng.normal(0, 1, size=n)
        estimates.append(np.std(sample))
    print(f'n={n:4d} ->', np.round(estimates, 3),
          'mean std =', round(float(np.mean(estimates)), 3))


## 2. ✏️ Implement z-score standardization `zscore(x)`

`(x − mean) / standard deviation`. After standardizing, mean ≈ 0 and standard deviation ≈ 1.

**Line of reasoning**:
1. Compute the mean, `mu = x.mean()` — after subtracting it, the data is centered on 0 and no longer leans to either side
2. Compute the standard deviation, `sigma = x.std()` — after dividing by it, deviations are expressed in units of "so many σ", so features in different units can be compared directly
3. Return `(x - mu) / sigma` — NumPy broadcasting makes this work for an `x` of any shape (such as an `(80, T)` mel matrix)

**Reference input and output**: `x=[2,4,4,4,5,5,7,9]` → mean=5, std≈2, z-score≈[-1.5, -0.5, -0.5, -0.5, 0, 0, 1, 2]

> 💡 Need a hint? Once you've done the exercise, compare with the reference implementation in the `solutions/` folder.


### Before writing `zscore`, pin down three things

- input: a 1-D array `x` of shape `(N,)`, any range of values
- key steps: `mean = x.mean()`, `std = x.std()`, `z = (x - mean) / std`
- return: an array of the same shape as `x`, with `z.mean() ≈ 0` and `z.std() ≈ 1` guaranteed

In [ ]:
def zscore(x):
    # ✏️ TODO: return the standardized array
    raise NotImplementedError("TODO: return the standardized array (x - μ) / σ")


In [ ]:
z = zscore(np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]))
print('mean after standardizing ≈', round(float(z.mean()), 6), ' standard deviation ≈', round(float(z.std()), 6))
assert abs(z.mean()) < 1e-9 and abs(z.std() - 1.0) < 1e-9
print('✅ Passed: you can standardize features now.')

## 3. A look at the distribution (histogram)

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
plt.hist(rng.normal(0, 1, 5000), bins=40)
plt.title('histogram of 5000 samples'); plt.show()

**🔗 Aurora connection**: z-score standardization is a **preprocessing step** in the `src/aurora/` feature pipeline, applied outside the feature extractor — `mfcc()` itself only does `mel_spectrogram → power_to_db → dct_ii`, with no built-in standardization. The actual training pipeline, after calling `mfcc()` and before feeding the model, applies `z = (F - F.mean()) / F.std()` to the batch features to remove the absolute energy differences between frequency bands.

Why does skipping this step cause trouble? Take a simple example:
- feature A: range [-1000, 1000] (not standardized)
- feature B: range [-0.1, 0.1] (not standardized)

When the model takes these two features as input and computes gradients directly, feature A dominates the gradient (because its numbers are big), and feature B's gradient gets "drowned out", so nothing is learned from it. After standardizing, both features live on the same scale (roughly around [-1, 1]), and the gradient can treat them fairly.

PyTorch's `nn.BatchNorm1d` normalizes with a mean/variance estimated on the fly across the mini-batch dimension; `nn.LayerNorm` normalizes across the feature dimension of a single sample — both are generalizations of the z-score with learnable parameters γ/β. The parameter γ controls the scaling and β the shift, letting the network decide for itself, after standardizing, how much each feature matters. Skip standardization (or pick the wrong axis), and gradients easily explode along high-variance feature dimensions or vanish along low-variance ones.

## Parameter experiment: z-scores along two different axes

Take a batch of mel features (shape `(80, T)`: 80 frequency bands, T frames) and standardize it in two different ways.

**First, get clear on what axis means**:
- the mel matrix has shape `(80, 200)`, i.e. 80 rows and 200 columns
- `axis=0` is the "row axis" (collapses the row direction) → operates along the frequency-band dimension → result has shape `(200,)` or `(1, 200)`
- `axis=1` is the "column axis" (collapses the column direction) → operates along the time dimension → result has shape `(80,)` or `(80, 1)`

```
F.shape = (80, 200)   # 80 frequency bands, 200 frames

F.mean(axis=0)  →  shape (200,)   # mean of each column (collapsing the band dimension)
F.mean(axis=1)  →  shape (80,)    # mean of each row (collapsing the time dimension)

# add keepdims=True to keep the dimension for broadcasting
F.mean(axis=0, keepdims=True)  →  shape (1, 200)
F.mean(axis=1, keepdims=True)  →  shape (80, 1)
```

**The two choices of standardization**:
- **along the time axis, `axis=1`**: `(F - F.mean(axis=1, keepdims=True)) / F.std(axis=1, keepdims=True)`
  each frequency band is standardized independently over its own time series, removing the absolute energy differences between bands
- **along the feature axis, `axis=0`**: `(F - F.mean(axis=0, keepdims=True)) / F.std(axis=0, keepdims=True)`
  each frame is standardized across its 80 bands, removing the overall energy differences between whole frames

The code below uses a small simulated matrix `rng.normal(0, 1, size=(4, 6))` (4 bands, 6 frames) to check:
- after `axis=1`, every row (every band) has mean ≈ 0 and std ≈ 1; each row is handled independently
- after `axis=0`, every column (every frame) has mean ≈ 0 and std ≈ 1; each column is handled independently

In [ ]:
import numpy as np

# Build a simulated mel feature matrix: (4 bands, 6 frames)
rng = np.random.default_rng(7)
mel = rng.normal(0, 1, size=(4, 6))
mel[2, :] += 50   # band 3 has abnormally high energy (simulating a baseline offset in one band)
mel[0, :] *= 10   # band 1 has a large variance

print("Original mel matrix (shape=(4,6)):")
print(mel)
print()

# Method 1: standardize along axis=1 (the time axis) — each row handled independently
mel_z1 = (mel - mel.mean(axis=1, keepdims=True)) / (mel.std(axis=1, keepdims=True) + 1e-8)
print("Method 1: after standardizing along axis=1 (each row independently):")
print(mel_z1)
print("Row means:", np.round(mel_z1.mean(axis=1), 2), " (should be ≈[0, 0, 0, 0])")
print("Row standard deviations:", np.round(mel_z1.std(axis=1), 2), " (should be ≈[1, 1, 1, 1])")
print()

# Method 2: standardize along axis=0 (the feature axis) — each column handled independently
mel_z0 = (mel - mel.mean(axis=0, keepdims=True)) / (mel.std(axis=0, keepdims=True) + 1e-8)
print("Method 2: after standardizing along axis=0 (each column independently):")
print(mel_z0)
print("Column means:", np.round(mel_z0.mean(axis=0), 2), " (should be ≈[0, 0, 0, 0, 0, 0])")
print("Column standard deviations:", np.round(mel_z0.std(axis=0), 2), " (should be ≈[1, 1, 1, 1, 1, 1])")
print()

print("💡 Key observations:")
print("   - axis=1: row 3 (high energy) comes back to a normal amplitude after standardizing, in line with the other rows")
print("   - axis=0: the relationships between bands within each frame are kept, but band amplitudes can differ from frame to frame")
print("   - audio applications usually choose axis=1, because we want to flatten baseline differences between microphones/hardware while keeping how each band changes over time")

### 🔧 NumPy tricks explained: `keepdims=True` and `+ 1e-8`

The cell above used two common NumPy idioms: `keepdims=True` and `+ 1e-8`. At first they look like "magic", so let's explain them properly.

#### Why `keepdims=True` is needed

**The problem**: when we take the mean or standard deviation of a multi-dimensional array along an axis, that dimension disappears.

```python
import numpy as np

F = np.array([[1, 2, 3],      # shape (2, 3)
              [4, 5, 6]])

print(F.mean(axis=1))           # mean along the rows (axis=1); the result has shape (2,)
# output: [2. 5.]

print(F.mean(axis=1, keepdims=True))  # with keepdims=True, the shape stays (2, 1)
# output: [[2.] [5.]]
```

**Why does this matter?** Broadcasting rules!

```python
# without keepdims: shape (2,3) - (2,) errors or broadcasts strangely
result = F - F.mean(axis=1)  # ❌ tries to broadcast [2, 5] to (2,3) — error!

# with keepdims: shape (2,3) - (2,1) broadcasts correctly to (2,3)
result = F - F.mean(axis=1, keepdims=True)  # ✅ (2,1) expands automatically to (2,3)
```

The broadcasting rule: line up the last dimensions; wherever one side has size 1, it is stretched to the other side's size. So when `(2,1)` meets `(2,3)`, the 1 expands to 3, and you get row-by-row subtraction.

#### `+ 1e-8` for numerical stability

**The problem**: if some dimension of the data has a standard deviation of exactly 0 (all values the same), then `(x - mean) / std` divides by zero.

```python
data = np.array([[1, 1, 1],     # this row is all 1s, std=0
                 [1, 2, 3]])

data_normalized = (data - data.mean(axis=1, keepdims=True)) / data.std(axis=1, keepdims=True)
# the first row comes out as [0/0, 0/0, 0/0] → [nan, nan, nan] ❌
```

**The fix**: add a tiny constant `1e-8` (that is, $10^{-8}$) to the denominator to prevent division by zero:

```python
eps = 1e-8
data_normalized = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + eps)
# first row: [0 / (0 + 1e-8), ...] = [0, 0, 0]  ✅ no longer nan
```

**Why `1e-8` and not some other value?** It's an empirical choice:
- too large (say 0.1): it changes the denominator and loses the real standard-deviation information
- too small (say 1e-20): it gets swallowed by rounding error in floating-point arithmetic
- `1e-8` is small enough for most applications (it doesn't affect the result) and large enough (not lost to rounding)

**Scientific notation cheat sheet**:
- `1e-6` = `0.000001` = one millionth
- `1e-8` = `0.00000001` = one hundred-millionth
- `1e-9` = `0.000000001` = one billionth

## Parameter experiment: z-scores along two different axes

Take a batch of mel features (shape `(80, T)`: 80 frequency bands, T frames) and standardize it in two different ways:

- **along the time axis, `axis=1`**: `(F - F.mean(axis=1, keepdims=True)) / F.std(axis=1, keepdims=True)`
  each frequency band is standardized independently over its own time series, removing the absolute energy differences between bands
- **along the feature axis, `axis=0`**: `(F - F.mean(axis=0, keepdims=True)) / F.std(axis=0, keepdims=True)`
  each frame is standardized across its 80 bands, removing the overall energy differences between whole frames

Generate simulated features with `rng.normal(0, 1, (80, 100))` and check: after `axis=1`, `result.mean(axis=1)` is all close to 0; after `axis=0`, `result.mean(axis=0)` is all close to 0. The choice of axis directly shapes the structure of the information the network sees — the first emphasizes how each band changes over time, the second the relative relationships between bands within each frame.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
mel = rng.normal(0, 1, size=(4, 8))
mel[2, :] += 50   # simulate a baseline offset in MFCC band 3

mel_z = (mel - mel.mean(axis=1, keepdims=True)) / (mel.std(axis=1, keepdims=True) + 1e-8)

print('row means before standardizing:', np.round(mel.mean(axis=1), 1))
print('row means after standardizing :', np.round(mel_z.mean(axis=1), 1))   # all ≈0
print('row std after standardizing   :', np.round(mel_z.std(axis=1), 1))    # all ≈1
print('→ after the z-score, row 3 (offset +50) has exactly the same amplitude as the other rows.')


### 📋 Checking multi-dimensional standardization more deeply: what happens on the other axis

A student asks: "After standardizing, what do the mean and standard deviation along the **other axis** become?"

Good question. Let's check by hand with a small matrix.

Suppose a matrix M of shape (2, 3):
```
M = [[1, 2, 3],
     [4, 5, 6]]
```

**After standardizing along axis=1 (the row direction)**:
```
M_z = (M - M.mean(axis=1, keepdims=True)) / (M.std(axis=1, keepdims=True) + eps)
    = (M - [[2], [5]]) / [[0.816], [0.816]]  # row 1 mean=2, row 2 mean=5; both have std=√(2/3)≈0.816
    ≈ [[−1.22, 0, 1.22],  # row 1 after standardizing
       [−1.22, 0, 1.22]]  # row 2 after standardizing
```

Now check:
- **each row** (axis=1): mean ≈ 0, std ≈ 1 ✓ (the axis we standardized)
- **each column** (axis=0): mean ≈ ?

```python
M_z.mean(axis=0) = [(-1.22 + -1.22) / 2, (0 + 0) / 2, (1.22 + 1.22) / 2] = [-1.22, 0, 1.22]  # not 0!
M_z.std(axis=0) = ...                                                                         # not 1 either!
```

**The key point**: after standardizing, the statistics **along the standardized axis** become (0, 1), but the statistics **along the other axes** generally change — that's normal and expected.

## Wrap-up

You can now use `zscore(x)` to transform an array with any mean and variance to mean 0 and standard deviation 1. This transform is used directly in Aurora's feature pipeline: MFCC features go through it before they're sent to a classifier. BatchNorm and LayerNorm are learnable generalizations of the z-score, whose parameters γ/β let the network decide its own scaling and shift after standardizing. The next lesson (L29) introduces common probability distributions — uniform, Gaussian and Bernoulli — and how their parameters are estimated.

In [ ]:
# Quick check: a hand-written z-score gets closer to μ=0, σ=1 as the sample grows
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    x = rng.normal(5.0, 2.0, size=n)   # original data μ=5, σ=2
    z = (x - x.mean()) / x.std()        # z-score
    print(f'n={n:4d} -> after standardizing mean={z.mean():+.4f}, std={z.std():.4f}')
print('→ the bigger n is, the closer the z-scored mean is to 0 and the std to 1.')


---
⬇️ **Final check**: once you've read the wrap-up, do the whiteboard challenge below, then fill in the self-review.


## ✏️ Whiteboard challenge: descriptive statistics by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: x = [2, 4, 4, 4, 5, 5, 7, 9]; compute by hand:
- μ = ?
- σ² = ?
- σ = ?

**Question 2**: x[0]=2; use the z-score formula to compute z₀ = ? by hand

**Question 3**: for the standardized array z = zscore(x), z.mean() ≈ ? z.std() ≈ ?

**Question 4**: a mel feature matrix M has shape (80, T); apply a z-score along the time axis, axis=1:
the mean of each row (each frequency band) becomes ?, and each row's standard deviation becomes ?
Why is this more sensible than standardizing along axis=0?

When you've worked it out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

# Q1: μ=5, σ²=4, σ=2
mu_expected = 5.0
var_expected = 4.0
std_expected = 2.0
assert np.isclose(x.mean(), mu_expected, atol=1e-10)
assert np.isclose(x.var(), var_expected, atol=1e-10)
assert np.isclose(x.std(), std_expected, atol=1e-10)
print(f"Q1 ✅  μ={x.mean():.1f}, σ²={x.var():.1f}, σ={x.std():.1f}")

# Q2: z₀ = (2-5)/2 = -1.5
z0_expected = (2.0 - mu_expected) / std_expected
assert np.isclose(z0_expected, -1.5, atol=1e-10)
print(f"Q2 ✅  z₀=(2-5)/2={z0_expected:.2f}  (1.5 standard deviations below the mean)")

# Q3: zscore(x).mean()≈0, zscore(x).std()≈1
try:
    z = zscore(x)
    assert abs(z.mean()) < 1e-10, f"mean after standardizing should be ≈0, got {z.mean()}"
    assert abs(z.std() - 1.0) < 1e-10, f"std after standardizing should be ≈1, got {z.std()}"
    assert np.isclose(z[0], z0_expected, atol=1e-10), f"z[0] should be {z0_expected}, got {z[0]}"
    print(f"Q3 ✅  zscore(x).mean()={z.mean():.2e}≈0, std={z.std():.6f}≈1")
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement zscore() first, then run the answer-check cell")

# Q4: z-score along the time axis, each row mean→0, std→1
rng = np.random.default_rng(0)
M = rng.normal(3.0, 5.0, size=(80, 200))   # 80 bands × 200 frames
M_z = (M - M.mean(axis=1, keepdims=True)) / M.std(axis=1, keepdims=True)
assert np.allclose(M_z.mean(axis=1), 0.0, atol=1e-10), "each row's mean should be 0"
assert np.allclose(M_z.std(axis=1), 1.0, atol=1e-10), "each row's std should be 1"
print(f"Q4 ✅  (80,200) mel matrix standardized along axis=1: each row mean≈0, std≈1")
print("     Along the time axis is more sensible: it removes each band's DC offset while keeping how each band changes over time")
print("\n🎉 Descriptive-statistics whiteboard challenge passed! You've got the geometric meaning of z-score standardization.")

In [ ]:
# ✏️ Self-review for this lesson
l28_review = {
    "mean_var_std_formula":       None,  # know the μ/σ²/σ formulas and can compute them by hand? True/False
    "zscore_implemented":         None,  # zscore implemented and assertions pass? True/False
    "zscore_properties":          None,  # understand that after standardizing mean=0, std=1? True/False
    "axis_direction_rationale":   None,  # understand the difference between standardizing along time vs frequency? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l28_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l28_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L28 fully cleared! On to L29: common distributions')

---

→ **Next lesson** · [L29 · Common distributions](L29_distributions.ipynb)

> Next lesson covers **common distributions**: uniform, Gaussian, Bernoulli — PDF / CDF and sampling.